<a href="https://colab.research.google.com/github/debanjanhati2-boop/Basic-Chatbot/blob/main/AI_Product_Description_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random
import re
import json
from datetime import datetime
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

class ProductDescriptionGenerator:
    def __init__(self):
        self.platform_configs = self.load_platform_configs()
        self.description_templates = self.load_description_templates()
        self.feature_templates = self.load_feature_templates()
        self.benefit_templates = self.load_benefit_templates()
        self.cta_templates = self.load_cta_templates()
        self.generated_descriptions = []

    def load_platform_configs(self):
        """Load configuration for each e-commerce platform"""
        return {
            'amazon': {
                'short_max': 500,
                'long_max': 2000,
                'bullet_count': 5,
                'tone': ['professional', 'informative', 'benefit-focused'],
                'structure': ['hook', 'features', 'benefits', 'cta']
            },
            'flipkart': {
                'short_max': 400,
                'long_max': 1500,
                'bullet_count': 4,
                'tone': ['friendly', 'engaging', 'value-focused'],
                'structure': ['hook', 'features', 'value', 'cta']
            },
            'shopify': {
                'short_max': 300,
                'long_max': 1200,
                'bullet_count': 4,
                'tone': ['brand-focused', 'storytelling', 'premium'],
                'structure': ['story', 'features', 'benefits', 'cta']
            },
            'woocommerce': {
                'short_max': 400,
                'long_max': 1500,
                'bullet_count': 5,
                'tone': ['informative', 'technical'],
                'structure': ['overview', 'specifications', 'features', 'cta']
            },
            'social_media': {
                'short_max': 150,
                'long_max': 500,
                'bullet_count': 3,
                'tone': ['casual', 'fun', 'engaging'],
                'structure': ['hook', 'highlight', 'cta']
            }
        }

    def load_description_templates(self):
        """Load description templates for different structures"""
        return {
            'hook': [
                "Introducing {product_name} - the {adjective} {category} that's changing the way you {action}.",
                "Say hello to {product_name}, your new {category} companion for {benefit}.",
                "Meet {product_name}, the {adjective} solution designed for {target_audience}.",
                "Discover {product_name}, where {adjective} design meets {feature} performance."
            ],
            'overview': [
                "The {product_name} is a premium {category} that combines {feature1} with {feature2} to deliver {benefit}. Perfect for {target_audience} who demand the best.",
                "Crafted with {feature1} and engineered for {feature2}, the {product_name} sets a new standard in {category}. It's the ideal choice for {benefit}."
            ],
            'story': [
                "We created {product_name} because we believe {target_audience} deserve better. Every detail, from {feature1} to {feature2}, is designed to help you {benefit}.",
                "Born from a passion for {category}, {product_name} represents the perfect blend of innovation and craftsmanship. It's more than just a product - it's a lifestyle."
            ],
            'value': [
                "Get more value with {product_name}. Featuring {feature1}, {feature2}, and {feature3}, it's everything you need at a price that makes sense.",
                "{product_name} delivers exceptional value with its {feature1}, {feature2}, and {feature3}. Smart choice for smart buyers."
            ]
        }

    def load_feature_templates(self):
        """Load feature bullet point templates"""
        return [
            "✅ PREMIUM QUALITY: Made with {material} for lasting durability and performance",
            "✅ {FEATURE}: {description} for enhanced {benefit}",
            "✅ ADVANCED {feature}: Enjoy superior {aspect} with cutting-edge technology",
            "✅ PERFECT FOR {target}: Designed specifically for {target_audience}'s needs",
            "✅ EASY TO USE: Simple setup and intuitive controls for hassle-free experience",
            "✅ VERSATILE DESIGN: Perfect for {use_case1} and {use_case2}",
            "✅ ENERGY EFFICIENT: Saves {savings} while delivering maximum performance",
            "✅ PORTABLE & LIGHTWEIGHT: Easy to carry and use anywhere",
            "✅ WARRANTY INCLUDED: Backed by {warranty} warranty for peace of mind",
            "✅ VALUE FOR MONEY: Premium features at an affordable price"
        ]

    def load_benefit_templates(self):
        """Load benefit statement templates"""
        return {
            'short_term': [
                "instantly improve your {activity}",
                "see immediate results in {area}",
                "experience the difference from day one"
            ],
            'long_term': [
                "achieve lasting {benefit} for years to come",
                "build a sustainable {activity} routine",
                "invest in your long-term {goal}"
            ],
            'emotional': [
                "feel confident and in control every day",
                "enjoy peace of mind knowing you have the best",
                "experience the joy of effortless {activity}"
            ]
        }

    def load_cta_templates(self):
        """Load call-to-action templates"""
        return [
            "🛒 Order now and experience the {product_name} difference!",
            "🎁 Get yours today with free shipping!",
            "⭐ Join thousands of satisfied customers - Buy now!",
            "🔥 Limited stock available - Order yours today!",
            "✨ Click 'Add to Cart' and start your journey today!"
        ]

    def extract_keywords(self, product_data):
        """Extract SEO keywords from product data"""
        keywords = []

        # Extract from category and type
        if 'category' in product_data:
            keywords.append(product_data['category'].lower())

        if 'features' in product_data:
            for feature in product_data['features']:
                keywords.append(feature.lower())

        # Add modifier keywords
        modifiers = ['best', 'top', 'premium', 'quality', 'affordable']
        if 'category' in product_data:
            for mod in modifiers:
                keywords.append(f"{mod} {product_data['category'].lower()}")

        return list(set(keywords))

    def generate_short_description(self, product_data):
        """Generate short product description"""
        template = random.choice(self.description_templates['hook'])

        description = template.format(
            product_name=product_data.get('name', 'This Product'),
            category=product_data.get('category', 'product'),
            adjective=random.choice(['revolutionary', 'innovative', 'premium', 'amazing']),
            action=product_data.get('use_case', 'live and work'),
            benefit=product_data.get('main_benefit', 'better results'),
            target_audience=product_data.get('target_audience', 'everyone'),
            feature=random.choice(product_data.get('features', ['advanced technology']))
        )

        # Truncate if needed
        max_length = 300
        if len(description) > max_length:
            description = description[:max_length-3] + "..."

        return description

    def generate_bullet_points(self, product_data, count=5):
        """Generate feature bullet points"""
        bullets = []
        features = product_data.get('features', [])

        # Select templates
        templates = random.sample(self.feature_templates, min(count, len(self.feature_templates)))

        for i, template in enumerate(templates):
            # Get feature name
            if i < len(features):
                feature = features[i]
            else:
                feature = random.choice(features) if features else "Quality"

            # Format bullet
            bullet = template.format(
                feature=feature.upper(),
                FEATURE=feature.upper(),
                description=random.choice(['Delivers exceptional results', 'Provides superior performance', 'Ensures maximum reliability']),
                benefit=product_data.get('main_benefit', 'productivity'),
                aspect=random.choice(['performance', 'durability', 'efficiency']),
                target=product_data.get('target_audience', 'Everyone'),
                target_audience=product_data.get('target_audience', 'users'),
                use_case1=random.choice(['home', 'office', 'travel']),
                use_case2=random.choice(['outdoor', 'indoor', 'professional']),
                savings=random.choice(['up to 30% energy', 'significant costs', 'valuable time']),
                warranty=random.choice(['1-year', '2-year', 'lifetime']),
                material=random.choice(['premium materials', 'high-grade components', 'quality materials'])
            )
            bullets.append(bullet)

        return bullets

    def generate_long_description(self, product_data):
        """Generate long product description"""
        sections = []

        # Overview
        overview_template = random.choice(self.description_templates['overview'])
        features = product_data.get('features', ['premium quality', 'advanced design', 'reliable performance'])

        overview = overview_template.format(
            product_name=product_data.get('name', 'This Product'),
            category=product_data.get('category', 'product'),
            feature1=features[0] if len(features) > 0 else 'quality',
            feature2=features[1] if len(features) > 1 else 'design',
            benefit=product_data.get('main_benefit', 'outstanding results'),
            target_audience=product_data.get('target_audience', 'users')
        )
        sections.append(overview)

        # Features section
        sections.append(f"\n\n**Key Features:**\n" + "\n".join(self.generate_bullet_points(product_data, 3)))

        # Benefits section
        benefit = random.choice(self.benefit_templates['long_term'])
        sections.append(f"\n\n**Why Choose {product_data.get('name', 'This Product')}?**\n{benefit.format(benefit=product_data.get('main_benefit', 'excellent results'), activity='your daily routine', goal='success')}.")

        return "\n".join(sections)

    def generate_meta_description(self, product_data):
        """Generate SEO meta description"""
        name = product_data.get('name', 'Product')
        category = product_data.get('category', 'product')
        benefit = product_data.get('main_benefit', 'excellent results')

        templates = [
            f"Shop {name} - the best {category} for {benefit}. Premium quality, great value. Order now with free shipping!",
            f"Discover {name}, your ultimate {category} solution. Experience {benefit} with our top-rated product. Buy today!",
            f"{name} - Premium {category} designed for {benefit}. Trusted by thousands. Get yours with our special offer!"
        ]

        description = random.choice(templates)

        # Truncate to 160 characters
        if len(description) > 160:
            description = description[:157] + "..."

        return description

    def generate_cta(self, product_name):
        """Generate call-to-action"""
        template = random.choice(self.cta_templates)
        return template.format(product_name=product_name)

    def generate_description(self, product_data, platform='amazon'):
        """Generate complete product description"""
        if platform not in self.platform_configs:
            platform = 'amazon'

        config = self.platform_configs[platform]
        product_name = product_data.get('name', 'This Product')

        # Generate components
        short_description = self.generate_short_description(product_data)
        bullet_points = self.generate_bullet_points(product_data, config['bullet_count'])
        long_description = self.generate_long_description(product_data)
        meta_description = self.generate_meta_description(product_data)
        cta = self.generate_cta(product_name)
        keywords = self.extract_keywords(product_data)

        # Combine into full description
        full_description = f"""# {product_name}

## Short Description
{short_description}

## Key Features
""" + "\n".join(bullet_points) + f"""

## Full Description
{long_description}

## {cta}

**Keywords:** {', '.join(keywords[:10])}
"""

        description_data = {
            'product_name': product_name,
            'platform': platform,
            'category': product_data.get('category', 'General'),
            'short_description': short_description,
            'bullet_points': bullet_points,
            'long_description': long_description,
            'meta_description': meta_description,
            'cta': cta,
            'keywords': keywords,
            'full_description': full_description,
            'word_count': len(full_description.split()),
            'generated_at': datetime.now().isoformat()
        }

        self.generated_descriptions.append(description_data)
        return description_data

    def generate_ab_variants(self, product_data, platform='amazon', count=3):
        """Generate multiple variants for A/B testing"""
        variants = []
        for i in range(count):
            description = self.generate_description(product_data, platform)
            variants.append({
                'variant': f"Variant {chr(65+i)}",
                'short_description': description['short_description'],
                'cta': description['cta'],
                'word_count': description['word_count']
            })
        return variants

    def get_description_stats(self):
        """Get statistics about generated descriptions"""
        if not self.generated_descriptions:
            return "No descriptions generated yet"

        platform_counts = Counter([d['platform'] for d in self.generated_descriptions])
        avg_words = sum([d['word_count'] for d in self.generated_descriptions]) / len(self.generated_descriptions)

        return {
            'total_descriptions': len(self.generated_descriptions),
            'platform_distribution': dict(platform_counts),
            'average_word_count': round(avg_words, 1),
            'categories_covered': len(set([d['category'] for d in self.generated_descriptions]))
        }

# Usage Example
generator = ProductDescriptionGenerator()

print("🛍️ AI Product Description Generator")
print("="*60)

# Sample product data
product = {
    'name': "SmartWatch Pro X",
    'category': "Smartwatch",
    'features': [
        "Heart Rate Monitor",
        "GPS Tracking",
        "Water Resistant",
        "7-Day Battery",
        "AMOLED Display"
    ],
    'target_audience': "fitness enthusiasts",
    'main_benefit': "tracking your health and fitness goals",
    'use_case': "monitor health and stay connected"
}

# Generate description for Amazon
print("\n🎯 Generating Amazon Product Description...")
description = generator.generate_description(product, "amazon")

print(f"\n📌 Product: {description['product_name']}")
print(f"📊 Word Count: {description['word_count']}")
print(f"\n📄 Meta Description:\n{description['meta_description']}")
print(f"\n📝 Full Description:\n{description['full_description']}")

# Generate A/B variants
print("\n\n🔬 Generating A/B Test Variants...")
variants = generator.generate_ab_variants(product, "amazon", 3)

for v in variants:
    print(f"\n  {v['variant']}:")
    print(f"    Short: {v['short_description'][:100]}...")
    print(f"    CTA: {v['cta']}")

# Generate for different platforms
print("\n\n🌐 Generating for Different Platforms...")
print("="*60)

for platform in ['flipkart', 'shopify', 'social_media']:
    desc = generator.generate_description(product, platform)
    print(f"\n📱 {platform.upper()}")
    print(f"   Short: {desc['short_description'][:80]}...")
    print(f"   Bullets: {len(desc['bullet_points'])}")

# Get statistics
print("\n\n📊 Description Statistics:")
print("="*60)
stats = generator.get_description_stats()
for key, value in stats.items():
    print(f"  {key.replace('_', ' ').title()}: {value}")

🛍️ AI Product Description Generator

🎯 Generating Amazon Product Description...

📌 Product: SmartWatch Pro X
📊 Word Count: 189

📄 Meta Description:
Shop SmartWatch Pro X - the best Smartwatch for tracking your health and fitness goals. Premium quality, great value. Order now with free shipping!

📝 Full Description:
# SmartWatch Pro X

## Short Description
Say hello to SmartWatch Pro X, your new Smartwatch companion for tracking your health and fitness goals.

## Key Features
✅ WARRANTY INCLUDED: Backed by lifetime warranty for peace of mind
✅ PERFECT FOR fitness enthusiasts: Designed specifically for fitness enthusiasts's needs
✅ ADVANCED WATER RESISTANT: Enjoy superior performance with cutting-edge technology
✅ 7-DAY BATTERY: Delivers exceptional results for enhanced tracking your health and fitness goals
✅ PORTABLE & LIGHTWEIGHT: Easy to carry and use anywhere

## Full Description
Crafted with Heart Rate Monitor and engineered for GPS Tracking, the SmartWatch Pro X sets a new standar